# Compilation-Ready 10-Qubit Shor Circuit for Factoring 15 with `a = 11`

This notebook starts the compilation pipeline from the **generic** Shor architecture for factoring `N = 15` with base `a = 11`. It carries the bare circuit through the first simplification pass, expands the surviving modular-multiply block, lowers the result to a CX/Clifford+T basis, and produces the ASAP-scheduled physical handoff circuit.

All reusable helper functions for this walkthrough live in `Compilation/shor_compilation.py`, so the notebook remains purely demonstrative.

Why start from a 10-qubit circuit?
- The smaller 5- and 6-qubit circuits are useful demonstrations, but they are hand-specialized to the `N = 15` example and take advantage of the fact that the order of `11 mod 15` is `r = 2`.
- For compilation, we want the circuit architecture that does **not** assume the factors of `15` or the order `r` in advance. That means using the same space-optimized layout that would be used for a general 4-bit modulus.
- Since `15` is a 4-bit number (`n = 4`), the standard space-optimized Shor layout uses `2n + 2 = 10` qubits: `1` recycled phase qubit, `4` modular-value qubits, `4` arithmetic-workspace qubits, and `1` clean ancilla.
- This layout also requires `2n = 8` semiclassical phase-estimation rounds. That is the right starting point for compilation, because those controlled modular-arithmetic blocks are the objects that will later be decomposed, transpiled, scheduled, and finally protected by the Steane code.

For this specific `a = 11`, `N = 15` instance, seven of those eight rounds are immediately trivial because `a^(2^k) mod 15 = 1` for `k = 1, 2, ..., 7`.

In the first compilation pass, those seven controlled modular-exponentiation blocks simplify to the Identity and are removed. After that pass, the quantum circuit we actually carry forward keeps only the single nontrivial modular exponentiation: `ctrl-U^(2^0)`.

The final sections of this notebook lower that one surviving block as a bare physical circuit. The next notebook begins only when the construction becomes Steane-specific: magic-state injection, logical encoding, syndrome extraction, and the full physical Steane circuit.


In [1]:
from html import escape
import importlib
from pathlib import Path
import sys

from IPython.display import HTML, display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "Compilation" / "shor_compilation.py").exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break
else:
    raise ImportError("Could not locate Compilation/shor_compilation.py")

from Compilation import shor_compilation
shor_compilation = importlib.reload(shor_compilation)


def display_scrollable_text(text):
    text = escape(str(text))
    display(
        HTML(
            "<div style='display: block; width: 100%; max-width: 100%; box-sizing: border-box; overflow-x: auto; overflow-y: hidden; border: 1px solid #d0d7de; padding: 0.75rem; background: #ffffff;'>"
            "<div style='display: inline-block; min-width: max-content;'>"
            f"<pre style='display: inline-block; width: max-content; margin: 0; white-space: pre; font-family: Menlo, Consolas, monospace; font-size: 12px; line-height: 1.2;'>{text}</pre>"
            "</div>"
            "</div>"
        )
    )


def display_scrollable_text_circuit(qc, fold=-1):
    display_scrollable_text(qc.draw(output="text", fold=fold))


## Qubit Budget

The dictionary below makes the 10-qubit choice explicit and keeps the 11-qubit alternative visible for comparison.

In [2]:
shor_compilation.generic_space_optimized_budget(4)


{'n_bits': 4,
 'phase_estimation_rounds': 8,
 'recycled_phase_qubits': 1,
 'modular_value_register_qubits': 4,
 'arithmetic_register_qubits': 4,
 'clean_ancillas': {'Takahashi_Kunihiro_2n_plus_2': 1,
  'Beauregard_2n_plus_3': 2},
 'total_qubits': {'Takahashi_Kunihiro_2n_plus_2': 10,
  'Beauregard_2n_plus_3': 11}}

## Original 10-Qubit Circuit Before the First Pass

This is the full pre-compilation top-level circuit, matching the layout from the `qiskit_shor's` notebook. It keeps all eight semiclassical phase-estimation rounds, including the seven rounds that will later compile away because their modular exponentiation is just the Identity for `a = 11`.


In [3]:
original_qc = shor_compilation.build_original_10q_shor_15_layout()
display_scrollable_text_circuit(original_qc, fold=-1)


## First-Pass Reduced 10-Qubit Circuit

The generic `2n + 2` layout starts with eight semiclassical phase-estimation rounds, but for `a = 11` and `N = 15`, the `k = 7, 6, ..., 1` modular exponentiations are all multiplication by `1 mod 15`.

In the first compilation pass, those seven controlled blocks reduce to the Identity and are removed. The circuit below is therefore the reduced 10-qubit circuit that survives that pass, keeping only the single nontrivial modular exponentiation `ctrl-U^(2^0)`.

The remaining sections continue lowering this same bare circuit. Steane encoding does not start until the next notebook.


In [4]:
first_pass_qc = shor_compilation.build_first_pass_reduced_10q_shor_15_layout()
display_scrollable_text_circuit(first_pass_qc, fold=-1)


## First-Pass Compilation Summary

The full eight-round schedule is still the correct conceptual starting point, but the summary below makes the first-pass simplification explicit: rounds `k = 7` through `k = 1` are compiled away because they are all the Identity, leaving only the `k = 0` round.

After this point, the notebook lowers that surviving round from an opaque modular-exponentiation block into a concrete bare circuit and then schedules it for the Steane notebook to consume.


In [5]:
shor_compilation.first_pass_compilation_summary()


{'compiled_away_as_identity': [{'phase_bit': 7,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True},
  {'phase_bit': 6,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True},
  {'phase_bit': 5,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True},
  {'phase_bit': 4,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True},
  {'phase_bit': 3,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True},
  {'phase_bit': 2,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True},
  {'phase_bit': 1,
   'a^(2^k) mod N': 1,
   'controlled_operation': 'multiply by 1 mod 15',
   'compiled_away_in_first_pass': True}],
 'retained_after_first_pass': [{

## Reduced Physical Circuit With `ctrl-U^(2^0)` Expanded to Gates

The circuit below replaces the opaque `ctrl-U^(2^0)` block with a compact direct compilation of the full 4-bit map `x -> 11x mod 15`. The `|0000>` and `|1111>` fix-up is kept, and it now uses the clean arithmetic workspace and clean ancilla as scratch so the later CX-basis transpilation requires fewer non-Clifford gates.


In [7]:
gate_only_k0_qc = shor_compilation.build_first_pass_reduced_10q_shor_15_gate_only_k0_layout()
display_scrollable_text_circuit(gate_only_k0_qc, fold=-1)
gate_only_k0_qc.count_ops()


OrderedDict([('cx', 24),
             ('h', 2),
             ('mcx_o1', 2),
             ('ccx', 2),
             ('x', 1),
             ('barrier', 1),
             ('measure', 1)])

## Reduced Physical Circuit With `ctrl-U^(2^0)` Transpiled to CX Gates

The circuit below lowers the remaining `ccx` and `mcx` operations from the previous clean-workspace gate-only circuit into a Clifford+T plus CX basis. The assertion checks that no non-basis operations remain after transpilation.


In [8]:
from qiskit import transpile

cx_basis_gates = ["cx", "h", "t", "tdg", "s", "sdg", "x"]
cx_transpiled_k0_qc = transpile(
    gate_only_k0_qc,
    basis_gates=cx_basis_gates,
    optimization_level=3,
    seed_transpiler=0,
)

allowed_ops = set(cx_basis_gates) | {"barrier", "measure"}
remaining_non_basis_ops = sorted(set(cx_transpiled_k0_qc.count_ops()) - allowed_ops)
if remaining_non_basis_ops:
    raise AssertionError(f"Unexpected non-basis operations remain: {remaining_non_basis_ops}")

display_scrollable_text_circuit(cx_transpiled_k0_qc, fold=-1)
cx_transpiled_k0_qc.count_ops()


OrderedDict([('cx', 56),
             ('t', 24),
             ('tdg', 20),
             ('h', 18),
             ('x', 7),
             ('s', 4),
             ('barrier', 1),
             ('measure', 1)])

## ASAP-Scheduled Physical Handoff Circuit

The final bare-circuit step is to left-align the CX-basis circuit so each operation occurs in the earliest dependency-compatible layer. This is the Qiskit-side equivalent of applying an ASAP moment schedule such as `cirq.AlignLeft()` within each barrier-delimited segment: barriers are retained as essential boundaries, and operations are moved left only up to the nearest barrier while preserving qubit and classical-bit dependencies.

The resulting `scheduled_physical_k0_qc` is the handoff object for `shor_15_steane_encoding.ipynb`. It is still an unencoded physical circuit.

The notebook also saves this handoff as a QPY artifact so the Steane notebook can load it even if it is run in a different kernel.


In [9]:
scheduled_physical_k0_qc = shor_compilation.asap_schedule_circuit(cx_transpiled_k0_qc)
asap_schedule_metadata = scheduled_physical_k0_qc.metadata["asap_schedule"]

print(f"ASAP left-aligned physical handoff layers: {asap_schedule_metadata['scheduled_layer_count']}")
print(f"ASAP barrier policy: {asap_schedule_metadata['barrier_policy']}")
scheduled_artifact_path = shor_compilation.save_scheduled_physical_k0_circuit(scheduled_physical_k0_qc)
print(f"saved scheduled physical K0 QPY artifact: {scheduled_artifact_path}")
display_scrollable_text_circuit(scheduled_physical_k0_qc, fold=-1)
scheduled_physical_k0_qc.count_ops()


ASAP left-aligned physical handoff layers: 86
ASAP barrier policy: preserved_as_segment_boundaries
saved scheduled physical K0 QPY artifact: /Users/salahedeen/COS583_Project/Compilation/artifacts/scheduled_physical_k0_qc.qpy


OrderedDict([('cx', 56),
             ('t', 24),
             ('tdg', 20),
             ('h', 18),
             ('x', 7),
             ('s', 4),
             ('barrier', 1),
             ('measure', 1)])